In [3]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [4]:
!pip install -q lettucedetect datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.7/124.7 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 109.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.4/87.4 kB 9.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.2 which is incompatible.


In [1]:
import numpy as np
import torch

print("NumPy:", np.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

NumPy: 2.5.2
GPU available: True
GPU: Tesla T4


In [2]:
import lettucedetect

print("LettuceDetect installed successfully!")

LettuceDetect installed successfully!


In [3]:
from datasets import load_dataset

dataset = load_dataset("wandb/RAGTruth-processed")

print(dataset)

README.md:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 22.3MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.88MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15090 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2700 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'query', 'context', 'output', 'task_type', 'quality', 'model', 'temperature', 'hallucination_labels', 'hallucination_labels_processed', 'input_str'],
        num_rows: 15090
    })
    test: Dataset({
        features: ['id', 'query', 'context', 'output', 'task_type', 'quality', 'model', 'temperature', 'hallucination_labels', 'hallucination_labels_processed', 'input_str'],
        num_rows: 2700
    })
})


In [4]:
print(dataset.keys())

dict_keys(['train', 'test'])


In [5]:
split_name = list(dataset.keys())[0]

print(split_name)
print(dataset[split_name].column_names)

train
['id', 'query', 'context', 'output', 'task_type', 'quality', 'model', 'temperature', 'hallucination_labels', 'hallucination_labels_processed', 'input_str']


In [6]:
example = dataset[split_name][0]

for key, value in example.items():
    print("\n---", key, "---")
    print(value)


--- id ---
0

--- query ---
Summarize the following news within 116 words:

--- context ---
Seventy years ago, Anne Frank died of typhus in a Nazi concentration camp at the age of 15. Just two weeks after her supposed death on March 31, 1945, the Bergen-Belsen concentration camp where she had been imprisoned was liberated -- timing that showed how close the Jewish diarist had been to surviving the Holocaust. But new research released by the Anne Frank House shows that Anne and her older sister, Margot Frank, died at least a month earlier than previously thought. Researchers re-examined archives of the Red Cross, the International Training Service and the Bergen-Belsen Memorial, along with testimonies of survivors. They concluded that Anne and Margot probably did not survive to March 1945 -- contradicting the date of death which had previously been determined by Dutch authorities. In 1944, Anne and seven others hiding in the Amsterdam secret annex were arrested and sent to the  Auschwi

In [7]:
for i in range(5):
    print("\n")
    print("=" * 70)
    print("EXAMPLE:", i)
    print("=" * 70)

    example = dataset[split_name][i]

    for key, value in example.items():
        print(f"\n{key}:")
        print(value)



EXAMPLE: 0

id:
0

query:
Summarize the following news within 116 words:

context:
Seventy years ago, Anne Frank died of typhus in a Nazi concentration camp at the age of 15. Just two weeks after her supposed death on March 31, 1945, the Bergen-Belsen concentration camp where she had been imprisoned was liberated -- timing that showed how close the Jewish diarist had been to surviving the Holocaust. But new research released by the Anne Frank House shows that Anne and her older sister, Margot Frank, died at least a month earlier than previously thought. Researchers re-examined archives of the Red Cross, the International Training Service and the Bergen-Belsen Memorial, along with testimonies of survivors. They concluded that Anne and Margot probably did not survive to March 1945 -- contradicting the date of death which had previously been determined by Dutch authorities. In 1944, Anne and seven others hiding in the Amsterdam secret annex were arrested and sent to the  Auschwitz-Birke

In [8]:
from lettucedetect.models.inference import HallucinationDetector

detector = HallucinationDetector(
    method="transformer",
    model_path="KRLabsOrg/lettucedect-v2-mmbert-base"
)

print("LettuceDetect loaded!")

config.json:   0%|          | 0.00/2.07k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 34.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.23GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

LettuceDetect loaded!


In [9]:
context = [
    "The Eiffel Tower is located in Paris. It was completed in 1889."
]

question = "Tell me about the Eiffel Tower."

answer = (
    "The Eiffel Tower is located in Paris. "
    "It was completed in 1889. "
    "It is 900 meters tall and was designed by Leonardo da Vinci."
)

predictions = detector.predict(
    context=context,
    question=question,
    answer=answer,
    output_format="spans"
)

print(predictions)

[{'start': 70, 'end': 85, 'confidence': 0.6680112481117249, 'text': '900 meters tall'}, {'start': 89, 'end': 102, 'confidence': 0.5113643407821655, 'text': ' was designed'}, {'start': 105, 'end': 123, 'confidence': 0.5982829332351685, 'text': ' Leonardo da Vinci'}]


In [10]:
example = dataset["train"][0]

context = [example["context"]]
question = example["query"]
answer = example["output"]

predictions = detector.predict(
    context=context,
    question=question,
    answer=answer,
    output_format="spans"
)

print("========== RAGTRUTH ==========")
print(example["hallucination_labels"])

print("\n========== LETTUCEDETECT ==========")
print(predictions)

========== RAGTRUTH ==========
[]

========== LETTUCEDETECT ==========
[]


In [11]:
example = dataset["train"][2]

context = [example["context"]]
question = example["query"]
answer = example["output"]

predictions = detector.predict(
    context=context,
    question=question,
    answer=answer,
    output_format="spans"
)

print("========== RAGTRUTH ==========")
print(example["hallucination_labels"])

print("\n========== RAGTRUTH PROCESSED ==========")
print(example["hallucination_labels_processed"])

print("\n========== LETTUCEDETECT ==========")

if len(predictions) == 0:
    print("No hallucination detected")
else:
    for pred in predictions:
        print("Detected:", pred["text"])
        print("Confidence:", round(pred["confidence"], 3))
        print()

========== RAGTRUTH ==========
[{"start": 636, "end": 653, "text": "February 7, 2022.", "meta": "EVIDENT CONFLICT\nOriginal: February 7 (1945)\nGenerated: February 7, 2022", "label_type": "Evident Conflict", "implicit_true": false, "due_to_null": false}, {"start": 871, "end": 969, "text": "has prompted the Anne Frank House to issue a corrected statement regarding the date of her passing", "meta": "HIGH INTRODUCTION OF NEW INFORMATION\nIt was not mentioned in the source content that the Anne Frank House issue a corrected statement regarding the date of her passing.", "label_type": "Evident Baseless Info", "implicit_true": false, "due_to_null": false}, {"start": 607, "end": 646, "text": "believed to have died before February 7", "meta": "EVIDENT CONFLICT\nOriginal:  exact dates of death for Anne and Margot remain unclear; sisters both had symptoms before February 7\nGenerated: believed to have died before February 7", "label_type": "Evident Conflict", "implicit_true": false, "due_to_null

In [12]:
from typing import List, Dict, Tuple


def span_overlap(pred_span: Tuple[int, int], gold_span: Tuple[int, int]) -> float:
    """
    Calculate overlap (IoU) between predicted and actual hallucination spans.
    """
    p_start, p_end = pred_span
    g_start, g_end = gold_span

    inter_start = max(p_start, g_start)
    inter_end = min(p_end, g_end)

    intersection = max(0, inter_end - inter_start)

    union = max(p_end, g_end) - min(p_start, g_start)

    if union == 0:
        return 0.0

    return intersection / union


def match_spans(predictions, gold_spans, overlap_threshold=0.3):
    """
    Match LettuceDetect predictions with RAGTruth hallucination spans.
    """

    matched = []
    used_gold_idx = set()
    unmatched_preds = []

    for pred in predictions:

        pred_range = (pred["start"], pred["end"])

        best_score = 0.0
        best_gold_idx = None

        for i, gold in enumerate(gold_spans):

            if i in used_gold_idx:
                continue

            gold_range = (gold["start"], gold["end"])

            score = span_overlap(pred_range, gold_range)

            if score > best_score:
                best_score = score
                best_gold_idx = i

        if best_score >= overlap_threshold and best_gold_idx is not None:

            matched.append(
                (pred, gold_spans[best_gold_idx], best_score)
            )

            used_gold_idx.add(best_gold_idx)

        else:
            unmatched_preds.append(pred)

    unmatched_golds = [
        g for i, g in enumerate(gold_spans)
        if i not in used_gold_idx
    ]

    return matched, unmatched_preds, unmatched_golds


def evaluate_dataset(all_predictions, all_gold_spans, overlap_threshold=0.3):

    total_tp = 0
    total_fp = 0
    total_fn = 0

    for preds, golds in zip(all_predictions, all_gold_spans):

        matched, unmatched_preds, unmatched_golds = match_spans(
            preds,
            golds,
            overlap_threshold
        )

        total_tp += len(matched)
        total_fp += len(unmatched_preds)
        total_fn += len(unmatched_golds)

    precision = (
        total_tp / (total_tp + total_fp)
        if (total_tp + total_fp) > 0
        else 0
    )

    recall = (
        total_tp / (total_tp + total_fn)
        if (total_tp + total_fn) > 0
        else 0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    return {
        "precision": round(precision, 3),
        "recall": round(recall, 3),
        "f1": round(f1, 3),
        "total_tp": total_tp,
        "total_fp": total_fp,
        "total_fn": total_fn
    }


print("Evaluation functions loaded successfully!")

Evaluation functions loaded successfully!


In [13]:
import ast

example = dataset["train"][2]

context = [example["context"]]
question = example["query"]
answer = example["output"]

# Run LettuceDetect
predictions = detector.predict(
    context=context,
    question=question,
    answer=answer,
    output_format="spans"
)

# RAGTruth stores hallucination_labels as a STRING
gold_spans_raw = example["hallucination_labels"]

# Convert string -> actual Python list of dictionaries
import json

gold_spans = json.loads(gold_spans_raw)

print("Predictions:")
print(predictions)

print("\nGold spans:")
print(gold_spans)

print("\nGold type:", type(gold_spans))

# Evaluate
results = evaluate_dataset(
    [predictions],
    [gold_spans],
    overlap_threshold=0.3
)

print("\n========== TEST EVALUATION ==========")
print("Precision:", results["precision"])
print("Recall:   ", results["recall"])
print("F1 Score: ", results["f1"])

print()
print("TP:", results["total_tp"])
print("FP:", results["total_fp"])
print("FN:", results["total_fn"])

Predictions:
[{'start': 628, 'end': 652, 'confidence': 0.8607193827629089, 'text': ' before February 7, 2022'}]

Gold spans:
[{'start': 636, 'end': 653, 'text': 'February 7, 2022.', 'meta': 'EVIDENT CONFLICT\nOriginal: February 7 (1945)\nGenerated: February 7, 2022', 'label_type': 'Evident Conflict', 'implicit_true': False, 'due_to_null': False}, {'start': 871, 'end': 969, 'text': 'has prompted the Anne Frank House to issue a corrected statement regarding the date of her passing', 'meta': 'HIGH INTRODUCTION OF NEW INFORMATION\nIt was not mentioned in the source content that the Anne Frank House issue a corrected statement regarding the date of her passing.', 'label_type': 'Evident Baseless Info', 'implicit_true': False, 'due_to_null': False}, {'start': 607, 'end': 646, 'text': 'believed to have died before February 7', 'meta': 'EVIDENT CONFLICT\nOriginal:  exact dates of death for Anne and Margot remain unclear; sisters both had symptoms before February 7\nGenerated: believed to have d

In [14]:
example = dataset["train"][2]

print("GOLD SPANS:")
print(example["hallucination_labels"])

print("\nTYPE:")
print(type(example["hallucination_labels"]))

print("\nFIRST ITEM:")
print(example["hallucination_labels"][0])

print("\nFIRST ITEM TYPE:")
print(type(example["hallucination_labels"][0]))

GOLD SPANS:
[{"start": 636, "end": 653, "text": "February 7, 2022.", "meta": "EVIDENT CONFLICT\nOriginal: February 7 (1945)\nGenerated: February 7, 2022", "label_type": "Evident Conflict", "implicit_true": false, "due_to_null": false}, {"start": 871, "end": 969, "text": "has prompted the Anne Frank House to issue a corrected statement regarding the date of her passing", "meta": "HIGH INTRODUCTION OF NEW INFORMATION\nIt was not mentioned in the source content that the Anne Frank House issue a corrected statement regarding the date of her passing.", "label_type": "Evident Baseless Info", "implicit_true": false, "due_to_null": false}, {"start": 607, "end": 646, "text": "believed to have died before February 7", "meta": "EVIDENT CONFLICT\nOriginal:  exact dates of death for Anne and Margot remain unclear; sisters both had symptoms before February 7\nGenerated: believed to have died before February 7", "label_type": "Evident Conflict", "implicit_true": false, "due_to_null": false}]

TYPE:
<

In [15]:
print("PREDICTIONS:")
print(predictions)

print("\nPREDICTIONS TYPE:")
print(type(predictions))

if len(predictions) > 0:
    print("\nFIRST PREDICTION:")
    print(predictions[0])
    print(type(predictions[0]))

PREDICTIONS:
[{'start': 628, 'end': 652, 'confidence': 0.8607193827629089, 'text': ' before February 7, 2022'}]

PREDICTIONS TYPE:
<class 'list'>

FIRST PREDICTION:
{'start': 628, 'end': 652, 'confidence': 0.8607193827629089, 'text': ' before February 7, 2022'}
<class 'dict'>


In [16]:
import json

all_predictions = []
all_gold_spans = []

NUM_EXAMPLES = len(dataset["test"])

for i in range(NUM_EXAMPLES):

    example = dataset["test"][i]

    context = [example["context"]]
    question = example["query"]
    answer = example["output"]

    # LettuceDetect prediction
    predictions = detector.predict(
        context=context,
        question=question,
        answer=answer,
        output_format="spans"
    )

    # RAGTruth gold labels
    gold_spans_raw = example["hallucination_labels"]

    # Convert JSON string -> Python list
    gold_spans = json.loads(gold_spans_raw)

    all_predictions.append(predictions)
    all_gold_spans.append(gold_spans)

    print(f"Completed {i + 1}/{NUM_EXAMPLES}")

print("\nAll examples completed!")

Completed 1/2700
Completed 2/2700
Completed 3/2700
Completed 4/2700
Completed 5/2700
Completed 6/2700
Completed 7/2700
Completed 8/2700
Completed 9/2700
Completed 10/2700
Completed 11/2700
Completed 12/2700
Completed 13/2700
Completed 14/2700
Completed 15/2700
Completed 16/2700
Completed 17/2700
Completed 18/2700
Completed 19/2700
Completed 20/2700
Completed 21/2700
Completed 22/2700
Completed 23/2700
Completed 24/2700
Completed 25/2700
Completed 26/2700
Completed 27/2700
Completed 28/2700
Completed 29/2700
Completed 30/2700
Completed 31/2700
Completed 32/2700
Completed 33/2700
Completed 34/2700
Completed 35/2700
Completed 36/2700
Completed 37/2700
Completed 38/2700
Completed 39/2700
Completed 40/2700
Completed 41/2700
Completed 42/2700
Completed 43/2700
Completed 44/2700
Completed 45/2700
Completed 46/2700
Completed 47/2700
Completed 48/2700
Completed 49/2700
Completed 50/2700
Completed 51/2700
Completed 52/2700
Completed 53/2700
Completed 54/2700
Completed 55/2700
Completed 56/2700
C

In [17]:
results = evaluate_dataset(
    all_predictions,
    all_gold_spans,
    overlap_threshold=0.3
)

print("========== FULL DATASET EVALUATION ==========")

print("Precision:", results["precision"])
print("Recall:   ", results["recall"])
print("F1 Score: ", results["f1"])

print()
print("True Positives: ", results["total_tp"])
print("False Positives:", results["total_fp"])
print("False Negatives:", results["total_fn"])

========== FULL DATASET EVALUATION ==========
Precision: 0.447
Recall:    0.434
F1 Score:  0.44

True Positives:  665
False Positives: 823
False Negatives: 868


In [18]:
for i in range(10):
    print(f"\n========== Example {i} ==========")
    print("LettuceDetect predictions:")
    print(all_predictions[i])

    print("RAGTruth gold spans:")
    print(all_gold_spans[i])

    print("Number predicted:", len(all_predictions[i]))
    print("Number gold:", len(all_gold_spans[i]))


========== Example 0 ==========
LettuceDetect predictions:
[]
RAGTruth gold spans:
[]
Number predicted: 0
Number gold: 0

========== Example 1 ==========
LettuceDetect predictions:
[]
RAGTruth gold spans:
[]
Number predicted: 0
Number gold: 0

========== Example 2 ==========
LettuceDetect predictions:
[]
RAGTruth gold spans:
[]
Number predicted: 0
Number gold: 0

========== Example 3 ==========
LettuceDetect predictions:
[]
RAGTruth gold spans:
[]
Number predicted: 0
Number gold: 0

========== Example 4 ==========
LettuceDetect predictions:
[]
RAGTruth gold spans:
[{'start': 0, 'end': 106, 'text': 'Three women, including Keonna Thomas of Philadelphia, were charged with attempting to join ISIS this week.', 'meta': 'EVIDENT CONFLICT:\nOriginal: Three women, including Keonna Thomas of Philadelphia, were charged with attempting to join ISIS this week.\nAIGC: It is Keonna Thomas who is charged of attempting to join ISIS. Meanwhile, the other two were arrested and accused of planning to bui

In [19]:
results_010 = evaluate_dataset(
    all_predictions,
    all_gold_spans,
    overlap_threshold=0.1
)

print("========== FULL DATASET EVALUATION (threshold=0.1) ==========")
print("Precision:", results_010["precision"])
print("Recall:   ", results_010["recall"])
print("F1 Score: ", results_010["f1"])
print()
print("True Positives: ", results_010["total_tp"])
print("False Positives:", results_010["total_fp"])
print("False Negatives:", results_010["total_fn"])

========== FULL DATASET EVALUATION (threshold=0.1) ==========
Precision: 0.504
Recall:    0.489
F1 Score:  0.497

True Positives:  750
False Positives: 738
False Negatives: 783


In [20]:
false_positives_010 = []
false_negatives_010 = []

for i in range(len(dataset["test"])):

    example = dataset["test"][i]

    preds = all_predictions[i]
    golds = all_gold_spans[i]

    matched, unmatched_preds, unmatched_golds = match_spans(
        preds,
        golds,
        overlap_threshold=0.1
    )

    for pred in unmatched_preds:
        false_positives_010.append({
            "index": i,
            "id": example["id"],
            "context": example["context"],
            "answer": example["output"],
            "predicted_text": pred["text"],
            "pred_start": pred["start"],
            "pred_end": pred["end"],
            "confidence": pred.get("confidence", None)
        })

    for gold in unmatched_golds:
        false_negatives_010.append({
            "index": i,
            "id": example["id"],
            "context": example["context"],
            "answer": example["output"],
            "gold_text": gold["text"],
            "gold_start": gold["start"],
            "gold_end": gold["end"],
            "label_type": gold.get("label_type", "")
        })

print("False Positives (threshold=0.1):", len(false_positives_010))
print("False Negatives (threshold=0.1):", len(false_negatives_010))

False Positives (threshold=0.1): 738
False Negatives (threshold=0.1): 783


In [21]:
import random

random.seed(42)

sample_fp_010 = random.sample(
    false_positives_010,
    min(50, len(false_positives_010))
)

print("New FP sample (threshold=0.1):", len(sample_fp_010))

New FP sample (threshold=0.1): 50


In [22]:
import pandas as pd

rows_010 = []

for x in sample_fp_010:
    rows_010.append({
        "example_id": x["id"],
        "context": x["context"],
        "answer": x["answer"],
        "problem_span": x["predicted_text"],
        "actually_hallucination": "",   # fill in YES / NO / UNSURE by hand
        "notes": ""
    })

review_df_010 = pd.DataFrame(rows_010)

review_df_010.to_csv("fp_review_threshold_010.csv", index=False)
print("Saved: fp_review_threshold_010.csv")

Saved: fp_review_threshold_010.csv


In [23]:
with open("false_negatives_010.json", "w") as f:
    json.dump(false_negatives_010, f)

print("Exported:", len(false_negatives_010), "false negatives")

Exported: 783 false negatives


In [24]:
import json
import random
import pandas as pd

# --------------------------------------------------
# 1. Load the false negatives exported from the notebook
# --------------------------------------------------

with open("false_negatives_010.json") as f:
    false_negatives_010 = json.load(f)

print("Loaded false negatives:", len(false_negatives_010))


# --------------------------------------------------
# 2. Sample 100 for manual review
# --------------------------------------------------

random.seed(42)

sample_fn_100 = random.sample(
    false_negatives_010,
    min(100, len(false_negatives_010))
)

print("FN sample size:", len(sample_fn_100))


# --------------------------------------------------
# 3. Build review sheet
# --------------------------------------------------

rows_fn = []

for x in sample_fn_100:
    rows_fn.append({
        "example_id": x["id"],
        "context": x["context"],
        "answer": x["answer"],
        "missed_span": x["gold_text"],
        "label_type": x["label_type"],
        "category": "",     # fill in by hand
        "notes": ""
    })

fn_review_df = pd.DataFrame(rows_fn)

fn_review_df.to_csv("100_fn_cases.csv", index=False)
print("Saved: 100_fn_cases.csv")

Loaded false negatives: 783
FN sample size: 100
Saved: 100_fn_cases.csv


In [31]:
import re
import ast

def extract_structured_data(context: str) -> dict:
    match = re.search(r"\{.*\}", context, flags=re.DOTALL)
    if not match:
        return {}
    raw = match.group(0)
    try:
        data = ast.literal_eval(raw)
        if isinstance(data, dict):
            return data
    except (ValueError, SyntaxError):
        pass
    return {}


ATTRIBUTE_KEYWORDS = {
    "WiFi": ["wifi", "wi-fi", "internet access"],
    "RestaurantsReservations": ["reservation"],
    "OutdoorSeating": ["outdoor seating", "outdoor dining", "patio seating"],
    "RestaurantsTakeOut": ["takeout", "take-out", "take out"],
    "RestaurantsGoodForGroups": ["good for groups", "suitable for groups", "great for groups"],
    "Music": ["live music", "music performance", "music venue"],
    "BusinessParking": ["parking"],
}

NEGATION_PATTERNS = re.compile(
    r"\b(no|not|without|does not offer|doesn't offer|unavailable|none available)\b",
    re.IGNORECASE
)

HEDGE_PATTERNS = re.compile(
    r"\b(does not mention|doesn't mention|no information|not specify|"
    r"not specified|unclear whether|not clear if|not indicate|"
    r"doesn't indicate|does not indicate|not provide information|"
    r"no data|not stated|unknown whether|no details|"
    r"not provide (specific )?details|does not offer|doesn't offer|"
    r"does not provide|doesn't provide|lacks information|"
    r"no mention|not mentioned|not available information)\b",
    re.IGNORECASE
)


def check_boolean_attributes(attributes: dict, answer: str) -> list:
    flags = []
    answer_lower = answer.lower()

    for field, keywords in ATTRIBUTE_KEYWORDS.items():
        source_value = attributes.get(field, "MISSING")

        for kw in keywords:
            idx = answer_lower.find(kw)
            if idx == -1:
                continue

            window_start = max(0, idx - 60)
            window_end = min(len(answer), idx + len(kw) + 60)
            snippet = answer[window_start:window_end]

            if HEDGE_PATTERNS.search(snippet):
                break

            if source_value is None or source_value == "MISSING":
                flags.append({
                    "field": field,
                    "issue": "attribute-invention",
                    "detail": f"Answer makes a claim about '{field}' "
                              f"({kw!r}) but source value is unknown/None.",
                    "snippet": snippet,
                })
            else:
                source_is_negative = source_value in (False, "no", "none", None)
                mentions_negation = bool(NEGATION_PATTERNS.search(snippet))

                if source_is_negative and not mentions_negation:
                    flags.append({
                        "field": field,
                        "issue": "attribute-contradiction",
                        "detail": f"Source says '{field}'={source_value!r} "
                                  f"(negative/false) but answer's mention "
                                  f"of {kw!r} doesn't read as a negation.",
                        "snippet": snippet,
                    })
                elif not source_is_negative and mentions_negation:
                    flags.append({
                        "field": field,
                        "issue": "attribute-contradiction",
                        "detail": f"Source says '{field}'={source_value!r} "
                                  f"(positive) but answer negates {kw!r}.",
                        "snippet": snippet,
                    })
            break

    return flags


def _to_12h(hour_str: str) -> str:
    try:
        h, m = hour_str.split(":")
        h, m = int(h), int(m)
    except ValueError:
        return hour_str
    period = "AM" if h < 12 else "PM"
    h12 = h % 12
    if h12 == 0:
        h12 = 12
    return f"{h12}:{m:02d} {period}"


TIME_PATTERN = re.compile(r"\b(\d{1,2})(?::(\d{2}))?\s*(AM|PM|am|pm)\b")


def check_hours(hours_dict: dict, answer: str) -> list:
    flags = []
    if not hours_dict:
        return flags

    valid_times = set()
    for day_range in hours_dict.values():
        if not day_range:
            continue
        for segment in str(day_range).split(","):
            segment = segment.strip()
            if "-" not in segment:
                continue
            open_t, close_t = segment.split("-", 1)
            valid_times.add(_to_12h(open_t.strip()))
            valid_times.add(_to_12h(close_t.strip()))

    if not valid_times:
        return flags

    for m in TIME_PATTERN.finditer(answer):
        hour, minute, period = m.groups()
        minute = minute or "00"
        stated = f"{int(hour)}:{minute} {period.upper()}"

        if stated not in valid_times:
            window_start = max(0, m.start() - 40)
            window_end = min(len(answer), m.end() + 40)
            flags.append({
                "field": "hours",
                "issue": "hours-mismatch",
                "detail": f"Answer states {stated!r}, doesn't match source hours.",
                "snippet": answer[window_start:window_end],
            })

    return flags


def check_attributes(context: str, answer: str) -> list:
    data = extract_structured_data(context)
    if not data:
        return []
    attributes = data.get("attributes", {}) or {}
    hours = data.get("hours", {}) or {}
    flags = []
    flags.extend(check_boolean_attributes(attributes, answer))
    flags.extend(check_hours(hours, answer))
    return flags


print("check_attributes() defined and ready to use.")

check_attributes() defined and ready to use.


In [32]:
invention_only_predictions = []

for i in range(len(dataset["test"])):
    example = dataset["test"][i]
    context = example["context"]
    answer = example["output"]

    flags = check_attributes(context, answer)

    example_preds = []
    for f in flags:
        if f["issue"] not in ("attribute-invention", "hours-mismatch"):
            continue
        pos = answer.find(f["snippet"])
        if pos == -1:
            continue
        example_preds.append({
            "start": pos,
            "end": pos + len(f["snippet"]),
            "text": f["snippet"],
            "confidence": None,
            "source": "rule"
        })

    invention_only_predictions.append(example_preds)

total_invention_flags = sum(len(p) for p in invention_only_predictions)
print("Invention+hours-only flags (with hedge fix):", total_invention_flags)

Invention+hours-only flags (with hedge fix): 258


In [33]:
hybrid_predictions_narrow = []

for lettuce_preds, rule_preds in zip(all_predictions, invention_only_predictions):
    hybrid_predictions_narrow.append(lettuce_preds + rule_preds)

results_hybrid_narrow = evaluate_dataset(
    hybrid_predictions_narrow,
    all_gold_spans,
    overlap_threshold=0.1
)

print("========== NARROW HYBRID (with hedge fix) ==========")
print("Precision:", results_hybrid_narrow["precision"])
print("Recall:   ", results_hybrid_narrow["recall"])
print("F1 Score: ", results_hybrid_narrow["f1"])
print()
print("True Positives: ", results_hybrid_narrow["total_tp"])
print("False Positives:", results_hybrid_narrow["total_fp"])
print("False Negatives:", results_hybrid_narrow["total_fn"])

print()
print("========== COMPARISON ==========")
print(f"{'Metric':<12}{'Baseline':<12}{'Narrow(before)':<16}{'Narrow(fixed)':<12}")
print(f"{'Precision':<12}{results_010['precision']:<12}{0.456:<16}{results_hybrid_narrow['precision']:<12}")
print(f"{'Recall':<12}{results_010['recall']:<12}{0.537:<16}{results_hybrid_narrow['recall']:<12}")
print(f"{'F1':<12}{results_010['f1']:<12}{0.493:<16}{results_hybrid_narrow['f1']:<12}")

========== NARROW HYBRID (with hedge fix) ==========
Precision: 0.465
Recall:    0.53
F1 Score:  0.495

True Positives:  812
False Positives: 934
False Negatives: 721

========== COMPARISON ==========
Metric      Baseline    Narrow(before)  Narrow(fixed)
Precision   0.504       0.456           0.465       
Recall      0.489       0.537           0.53        
F1          0.497       0.493           0.495       


In [29]:
rule_fp_examples_v2 = []

for i in range(len(dataset["test"])):
    preds = invention_only_predictions[i]
    if not preds:
        continue
    golds = all_gold_spans[i]

    matched, unmatched_preds, _ = match_spans(preds, golds, overlap_threshold=0.1)

    for p in unmatched_preds:
        rule_fp_examples_v2.append({
            "index": i,
            "id": dataset["test"][i]["id"],
            "flagged_text": p["text"],
            "answer": dataset["test"][i]["output"],
        })

print("Total rule false positives (after hedge fix):", len(rule_fp_examples_v2))

import random
random.seed(7)  # different seed so we see a fresh batch this time
sample = random.sample(rule_fp_examples_v2, min(20, len(rule_fp_examples_v2)))

for s in sample:
    print(f"\n[{s['id']}] flagged: {s['flagged_text']!r}")

Total rule false positives (after hedge fix): 119

[7780] flagged: 'nformed.\n\nThe restaurant has attributes such as no business parking, no outdoor seating, and no WiFi. They do offer takeout and'

[7000] flagged: 'ta, the business does not offer BusinessParking, RestaurantsReservations, OutdoorSeating, WiFi, or RestaurantsTakeOut. However, it '

[8148] flagged: 'are no details on whether the business offers services like WiFi, outdoor seating, or is good for groups.'

[8670] flagged: ' not provide specific details on amenities such as business parking, reservations, outdoor seating, WiFi, takeout options, musi'

[6686] flagged: ' information about the business, including parking options, reservation policies, outdoor seating availability, WiFi accessibility,'

[6719] flagged: 'biance. Free Wi-Fi is available, and the restaurant accepts reservations and takeout orders. Overall, Mr. B Restaurant & Cafe is a '

[9936] flagged: 'oor seating only becomes available from 8:00 AM.'

[8560] 

In [ ]:
rule_predictions = []

for i in range(len(dataset["test"])):
    example = dataset["test"][i]
    context = example["context"]
    answer = example["output"]

    flags = check_attributes(context, answer)

    example_preds = []
    for f in flags:
        pos = answer.find(f["snippet"])
        if pos == -1:
            continue
        example_preds.append({
            "start": pos,
            "end": pos + len(f["snippet"]),
            "text": f["snippet"],
            "confidence": None,
            "source": "rule"
        })

    rule_predictions.append(example_preds)

total_rule_flags = sum(len(p) for p in rule_predictions)
print("Total rule-based flags across dataset:", total_rule_flags)

Total rule-based flags across dataset: 785


In [ ]:
hybrid_predictions = []

for lettuce_preds, rule_preds in zip(all_predictions, rule_predictions):
    hybrid_predictions.append(lettuce_preds + rule_preds)

print("Hybrid predictions built for", len(hybrid_predictions), "examples")

Hybrid predictions built for 2700 examples


In [ ]:
results_hybrid = evaluate_dataset(
    hybrid_predictions,
    all_gold_spans,
    overlap_threshold=0.1
)

print("========== HYBRID SYSTEM EVALUATION (threshold=0.1) ==========")
print("Precision:", results_hybrid["precision"])
print("Recall:   ", results_hybrid["recall"])
print("F1 Score: ", results_hybrid["f1"])
print()
print("True Positives: ", results_hybrid["total_tp"])
print("False Positives:", results_hybrid["total_fp"])
print("False Negatives:", results_hybrid["total_fn"])

========== HYBRID SYSTEM EVALUATION (threshold=0.1) ==========
Precision: 0.372
Recall:    0.552
F1 Score:  0.445

True Positives:  846
False Positives: 1427
False Negatives: 687


In [34]:
gap_fill_predictions = []

for lettuce_preds, rule_preds in zip(all_predictions, invention_only_predictions):
    if len(lettuce_preds) == 0:
        # LettuceDetect found nothing here — let the rule fill the gap
        gap_fill_predictions.append(lettuce_preds + rule_preds)
    else:
        # LettuceDetect already made predictions — trust it, don't add rule noise
        gap_fill_predictions.append(lettuce_preds)

results_gap_fill = evaluate_dataset(
    gap_fill_predictions,
    all_gold_spans,
    overlap_threshold=0.1
)

print("========== GAP-FILL HYBRID ==========")
print("Precision:", results_gap_fill["precision"])
print("Recall:   ", results_gap_fill["recall"])
print("F1 Score: ", results_gap_fill["f1"])
print()
print("True Positives: ", results_gap_fill["total_tp"])
print("False Positives:", results_gap_fill["total_fp"])
print("False Negatives:", results_gap_fill["total_fn"])

print()
print("========== FULL COMPARISON ==========")
print(f"{'Metric':<12}{'Baseline':<12}{'Narrow(fixed)':<16}{'Gap-fill':<12}")
print(f"{'Precision':<12}{results_010['precision']:<12}{results_hybrid_narrow['precision']:<16}{results_gap_fill['precision']:<12}")
print(f"{'Recall':<12}{results_010['recall']:<12}{results_hybrid_narrow['recall']:<16}{results_gap_fill['recall']:<12}")
print(f"{'F1':<12}{results_010['f1']:<12}{results_hybrid_narrow['f1']:<16}{results_gap_fill['f1']:<12}")

========== GAP-FILL HYBRID ==========
Precision: 0.495
Recall:    0.5
F1 Score:  0.498

True Positives:  767
False Positives: 782
False Negatives: 766

========== FULL COMPARISON ==========
Metric      Baseline    Narrow(fixed)   Gap-fill    
Precision   0.504       0.465           0.495       
Recall      0.489       0.53            0.5         
F1          0.497       0.495           0.498       


In [ ]:
# --------------------------------------------------
# Split rule flags: invention/hours only vs everything
# --------------------------------------------------

invention_only_predictions = []

for i in range(len(dataset["test"])):
    example = dataset["test"][i]
    context = example["context"]
    answer = example["output"]

    flags = check_attributes(context, answer)

    example_preds = []
    for f in flags:
        if f["issue"] not in ("attribute-invention", "hours-mismatch"):
            continue  # skip the coarse attribute-contradiction flags
        pos = answer.find(f["snippet"])
        if pos == -1:
            continue
        example_preds.append({
            "start": pos,
            "end": pos + len(f["snippet"]),
            "text": f["snippet"],
            "confidence": None,
            "source": "rule"
        })

    invention_only_predictions.append(example_preds)

total_invention_flags = sum(len(p) for p in invention_only_predictions)
print("Invention+hours-only flags:", total_invention_flags)
print("(compare to 785 total flags including contradiction check)")

Invention+hours-only flags: 317
(compare to 785 total flags including contradiction check)


In [ ]:
# --------------------------------------------------
# Build the narrower hybrid and re-evaluate
# --------------------------------------------------

hybrid_predictions_narrow = []

for lettuce_preds, rule_preds in zip(all_predictions, invention_only_predictions):
    hybrid_predictions_narrow.append(lettuce_preds + rule_preds)

results_hybrid_narrow = evaluate_dataset(
    hybrid_predictions_narrow,
    all_gold_spans,
    overlap_threshold=0.1
)

print("========== NARROW HYBRID (invention + hours only) ==========")
print("Precision:", results_hybrid_narrow["precision"])
print("Recall:   ", results_hybrid_narrow["recall"])
print("F1 Score: ", results_hybrid_narrow["f1"])
print()
print("True Positives: ", results_hybrid_narrow["total_tp"])
print("False Positives:", results_hybrid_narrow["total_fp"])
print("False Negatives:", results_hybrid_narrow["total_fn"])

print()
print("========== COMPARISON ==========")
print(f"{'Metric':<12}{'Baseline':<12}{'Full rule':<12}{'Narrow rule':<12}")
print(f"{'Precision':<12}{results_010['precision']:<12}{results_hybrid['precision']:<12}{results_hybrid_narrow['precision']:<12}")
print(f"{'Recall':<12}{results_010['recall']:<12}{results_hybrid['recall']:<12}{results_hybrid_narrow['recall']:<12}")
print(f"{'F1':<12}{results_010['f1']:<12}{results_hybrid['f1']:<12}{results_hybrid_narrow['f1']:<12}")

========== NARROW HYBRID (invention + hours only) ==========
Precision: 0.456
Recall:    0.537
F1 Score:  0.493

True Positives:  823
False Positives: 982
False Negatives: 710

========== COMPARISON ==========
Metric      Baseline    Full rule   Narrow rule 
Precision   0.504       0.372       0.456       
Recall      0.489       0.552       0.537       
F1          0.497       0.445       0.493       


In [ ]:
# --------------------------------------------------
# Sample some of the rule's flags that did NOT match
# any gold span, to see what's going wrong
# --------------------------------------------------

rule_fp_examples = []

for i in range(len(dataset["test"])):
    preds = invention_only_predictions[i]
    if not preds:
        continue
    golds = all_gold_spans[i]

    matched, unmatched_preds, _ = match_spans(preds, golds, overlap_threshold=0.1)

    for p in unmatched_preds:
        rule_fp_examples.append({
            "index": i,
            "id": dataset["test"][i]["id"],
            "flagged_text": p["text"],
            "answer": dataset["test"][i]["output"],
        })

print("Total rule false positives:", len(rule_fp_examples))

import random
random.seed(42)
sample = random.sample(rule_fp_examples, min(15, len(rule_fp_examples)))

for s in sample:
    print(f"\n[{s['id']}] flagged: {s['flagged_text']!r}")

Total rule false positives: 133

[7024] flagged: 'hours of operation are from 12:00 pm to 20:00 pm from Monday to Thursday, and from 11:00'

[6631] flagged: ' services. The data does not mention if reservations are accepted or if outdoor seating is '

[8557] flagged: 'staurant reservations, takeout service, WiFi availability, or whether the restaurant'

[8148] flagged: 'vices like WiFi, outdoor seating, or is good for groups.'

[7984] flagged: 'o 2pm every day, and offers takeout and outdoor seating. No reservations are available, but cus'

[7159] flagged: ' 10:0 PM, Wednesday and Thursday from 1:0 PM to 11:0 PM, and Friday from 12:0 PM to '

[7000] flagged: 'ta, the business does not offer BusinessParking, RestaurantsReservations, OutdoorSeatin'

[6998] flagged: 'r amenities such as WiFi, reservations, outdoor seating, or takeout. According to customer revi'

[9553] flagged: 'ion available about the availability of WiFi or takeout options.\n\nIn conclusion, Old'

[6634] flagged: 'busi

In [ ]:
import pandas as pd

df = pd.read_csv("categorized_errors.csv")

baseless_cases = df[
    df["category"] == "Baseless/unsupported info"
].copy()

print("Baseless/unsupported cases found:", len(baseless_cases))

baseless_cases["sub_pattern"] = ""   # you'll fill this in by hand
baseless_cases["notes"] = ""

baseless_cases.to_csv("30_baseless_cases.csv", index=False)
print("Saved: 30_baseless_cases.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'categorized_errors.csv'

In [ ]:
print("Examples evaluated:", len(all_predictions))

print(
    "Examples where LettuceDetect predicted something:",
    sum(len(x) > 0 for x in all_predictions)
)

print(
    "Total LettuceDetect predicted spans:",
    sum(len(x) for x in all_predictions)
)

print(
    "Total RAGTruth gold spans:",
    sum(len(x) for x in all_gold_spans)
)

Examples evaluated: 2700
Examples where LettuceDetect predicted something: 709
Total LettuceDetect predicted spans: 1488
Total RAGTruth gold spans: 1533


In [ ]:
import re

def text_overlap_ratio(pred_text: str, gold_text: str) -> float:
    """Rough token-overlap check, independent of character offsets."""
    p_tokens = set(re.findall(r"\w+", pred_text.lower()))
    g_tokens = set(re.findall(r"\w+", gold_text.lower()))
    if not p_tokens or not g_tokens:
        return 0.0
    return len(p_tokens & g_tokens) / len(p_tokens | g_tokens)


def diagnose_offset_alignment(all_predictions, all_gold_spans, answers, n_show=8):
    """
    Finds cases where pred/gold spans clearly describe the same text
    (high token overlap) but score low/zero on char-offset IoU.
    If these are common, you likely have an offset-basis mismatch,
    not a genuine detection miss.
    """
    suspects = []

    for ex_idx, (preds, golds) in enumerate(zip(all_predictions, all_gold_spans)):
        for pred in preds:
            for gold in golds:
                char_iou = span_overlap(
                    (pred["start"], pred["end"]),
                    (gold["start"], gold["end"]),
                )
                txt_overlap = text_overlap_ratio(
                    pred.get("text", ""), gold.get("text", "")
                )

                # High text similarity but low/no char overlap = red flag
                if txt_overlap >= 0.5 and char_iou < 0.3:
                    suspects.append({
                        "example_idx": ex_idx,
                        "pred_text": pred.get("text", ""),
                        "pred_span": (pred["start"], pred["end"]),
                        "gold_text": gold.get("text", ""),
                        "gold_span": (gold["start"], gold["end"]),
                        "char_iou": round(char_iou, 3),
                        "text_overlap": round(txt_overlap, 3),
                    })

    print(f"Found {len(suspects)} likely offset-mismatch cases "
          f"(high text similarity, low char IoU)\n")

    for s in suspects[:n_show]:
        ex = s["example_idx"]
        answer_text = answers[ex]
        print("=" * 70)
        print(f"Example {ex}")
        print(f"  Pred text:  {s['pred_text']!r}")
        print(f"  Pred span:  {s['pred_span']}")
        print(f"  Gold text:  {s['gold_text']!r}")
        print(f"  Gold span:  {s['gold_span']}")
        print(f"  char IoU={s['char_iou']}  text_overlap={s['text_overlap']}")
        # Show what characters the answer text actually has at each span,
        # so you can see the shift directly
        p_s, p_e = s["pred_span"]
        g_s, g_e = s["gold_span"]
        print(f"  answer[pred_span] = {answer_text[p_s:p_e]!r}")
        print(f"  answer[gold_span] = {answer_text[g_s:g_e]!r}")
        print()

    return suspects


# You'll need the raw answer text per example to run the offset check below.
# Rebuild it alongside your existing loop, or re-derive it here:
answers = [dataset["test"][i]["output"] for i in range(len(all_predictions))]

suspects = diagnose_offset_alignment(all_predictions, all_gold_spans, answers)

Found 8 likely offset-mismatch cases (high text similarity, low char IoU)

Example 1204
  Pred text:  '100 reviews'
  Pred span:  (283, 294)
  Gold text:  '100'
  Gold span:  (283, 286)
  char IoU=0.273  text_overlap=0.5
  answer[pred_span] = '100 reviews'
  answer[gold_span] = '100'

Example 1240
  Pred text:  ' casual setting'
  Pred span:  (938, 953)
  Gold text:  'casual'
  Gold span:  (389, 395)
  char IoU=0.0  text_overlap=0.5
  answer[pred_span] = ' casual setting'
  answer[gold_span] = 'casual'

Example 1415
  Pred text:  ' has a casual, classy atmosphere'
  Pred span:  (836, 868)
  Gold text:  'has a cozy atmosphere'
  Gold span:  (1016, 1037)
  char IoU=0.0  text_overlap=0.5
  answer[pred_span] = ' has a casual, classy atmosphere'
  answer[gold_span] = 'has a cozy atmosphere'

Example 1517
  Pred text:  ' provides outdoor seating'
  Pred span:  (405, 430)
  Gold text:  'outdoor seating,'
  Gold span:  (1176, 1192)
  char IoU=0.0  text_overlap=0.667
  answer[pred_span] = ' pro

In [ ]:
thresholds = [0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]

sweep_results = []
for t in thresholds:
    r = evaluate_dataset(all_predictions, all_gold_spans, overlap_threshold=t)
    sweep_results.append({"threshold": t, **r})
    print(f"threshold={t:>4}  P={r['precision']:.3f}  R={r['recall']:.3f}  "
          f"F1={r['f1']:.3f}  TP={r['total_tp']}  FP={r['total_fp']}  FN={r['total_fn']}")

threshold= 0.1  P=0.504  R=0.489  F1=0.497  TP=750  FP=738  FN=783
threshold=0.15  P=0.494  R=0.479  F1=0.487  TP=735  FP=753  FN=798
threshold= 0.2  P=0.480  R=0.466  F1=0.473  TP=714  FP=774  FN=819
threshold= 0.3  P=0.447  R=0.434  F1=0.440  TP=665  FP=823  FN=868
threshold= 0.4  P=0.407  R=0.395  F1=0.401  TP=606  FP=882  FN=927
threshold= 0.5  P=0.364  R=0.353  F1=0.358  TP=541  FP=947  FN=992
threshold= 0.6  P=0.321  R=0.311  F1=0.316  TP=477  FP=1011  FN=1056
threshold= 0.7  P=0.274  R=0.265  F1=0.269  TP=407  FP=1081  FN=1126


In [ ]:
false_positives = []
false_negatives = []

for i in range(len(dataset["test"])):

    example = dataset["test"][i]

    preds = all_predictions[i]
    golds = all_gold_spans[i]

    matched, unmatched_preds, unmatched_golds = match_spans(
        preds,
        golds,
        overlap_threshold=0.3
    )

    # False Positives
    for pred in unmatched_preds:
        false_positives.append({
            "index": i,
            "id": example["id"],
            "context": example["context"],
            "answer": example["output"],
            "predicted_text": pred["text"],
            "pred_start": pred["start"],
            "pred_end": pred["end"],
            "confidence": pred.get("confidence", None)
        })

    # False Negatives
    for gold in unmatched_golds:
        false_negatives.append({
            "index": i,
            "id": example["id"],
            "context": example["context"],
            "answer": example["output"],
            "gold_text": gold["text"],
            "gold_start": gold["start"],
            "gold_end": gold["end"],
            "label_type": gold.get("label_type", "")
        })

print("False Positives:", len(false_positives))
print("False Negatives:", len(false_negatives))

False Positives: 823
False Negatives: 868


In [ ]:
fp = false_positives[0]

print("========== FALSE POSITIVE ==========")

print("\nANSWER:")
print(fp["answer"])

print("\nLETTUCEDETECT WRONGLY FLAGGED:")
print(fp["predicted_text"])

print("\nCONFIDENCE:")
print(fp["confidence"])

========== FALSE POSITIVE ==========

ANSWER:
B.B. King, the legendary blues musician, was hospitalized for dehydration caused by his Type II diabetes. However, he has since been discharged and is now resting at home. The cause of his dehydration was attributed to his busy schedule and not drinking enough water. King is known for his hit songs such as "The Thrill Is Gone" and "There Must be a Better World Somewhere".

LETTUCEDETECT WRONGLY FLAGGED:
 he has since been discharged and is now resting

CONFIDENCE:
0.7028703689575195


In [ ]:
fn = false_negatives[0]

print("========== FALSE NEGATIVE ==========")

print("\nANSWER:")
print(fn["answer"])

print("\nRAGTRUTH SAYS THIS IS HALLUCINATED:")
print(fn["gold_text"])

print("\nHALLUCINATION TYPE:")
print(fn["label_type"])

========== FALSE NEGATIVE ==========

ANSWER:
Three women, including Keonna Thomas of Philadelphia, were charged with attempting to join ISIS this week. Thomas purchased a ticket to Barcelona but was arrested before her trip. Two other women, Noelle Velentzas and Asia Siddiqui, were arrested in New York for planning to build an explosive device. The FBI cited social media messages dating back to 2013 as evidence. This brings the total number of US citizens charged with supporting terrorism to over 30 in the past 18 months, with 18 of those cases involving ISIS.

RAGTRUTH SAYS THIS IS HALLUCINATED:
Three women, including Keonna Thomas of Philadelphia, were charged with attempting to join ISIS this week.

HALLUCINATION TYPE:
Evident Conflict


In [ ]:
fp = false_positives[0]

print("========== ORIGINAL SOURCE ==========")
print(fp["context"])

print("\n========== AI ANSWER ==========")
print(fp["answer"])

print("\n========== LETTUCEDETECT FLAGGED ==========")
print(fp["predicted_text"])

========== ORIGINAL SOURCE ==========
Blues legend B.B. King was hospitalized for dehydration, though the ailment didn't keep him out for long. King's dehydration was caused by his Type II diabetes, but he "is much better," his daughter, Claudette King, told the Los Angeles Times. The legendary guitarist and vocalist released a statement thanking those who have expressed their concerns. "I'm feeling much better and am leaving the hospital today," King said in a message Tuesday. Angela Moore, a publicist for Claudette King, said later in the day that he was back home resting and enjoying time with his grandchildren. "He was struggling before, and he is a trouper," Moore said. "He wasn't going to let his fans down." No more information on King's condition or where he was hospitalized was immediately available. B.B. is short for Blues Boy, part of the name he used as a Memphis disc jockey, the Beale Street Blues Boy. He was inducted into the Rock and Roll Hall of Fame in 1987, and has 30 

In [ ]:
import random

random.seed(42)

sample_fp = random.sample(
    false_positives,
    min(50, len(false_positives))
)

sample_fn = random.sample(
    false_negatives,
    min(50, len(false_negatives))
)

print("FP sample:", len(sample_fp))
print("FN sample:", len(sample_fn))
print("Total errors to analyze:", len(sample_fp) + len(sample_fn))

FP sample: 50
FN sample: 50
Total errors to analyze: 100


In [ ]:
import pandas as pd

rows = []

for x in sample_fp:
    rows.append({
        "error_type": "FP",
        "example_id": x["id"],
        "context": x["context"],
        "answer": x["answer"],
        "problem_span": x["predicted_text"],
        "ragtruth_type": "",
        "reason": ""
    })

for x in sample_fn:
    rows.append({
        "error_type": "FN",
        "example_id": x["id"],
        "context": x["context"],
        "answer": x["answer"],
        "problem_span": x["gold_text"],
        "ragtruth_type": x["label_type"],
        "reason": ""
    })

error_df = pd.DataFrame(rows)

error_df.head()

,error_type,example_id,context,answer,problem_span,ragtruth_type,reason
0,FP,13532,One difference between the FEHA (the Fair Empl...,"Based on the given passages, we can see that t...",twenty or more employees,,
1,FP,6277,"\n{'name': 'Mala Town', 'address': '6555 Parda...",Mala Town is a local Chinese restaurant locate...,to,,
2,FP,1112,An American citizen was wounded by gunfire Thu...,An American woman working at a medical school ...,The attack occurred despite a recent surge in...,,
3,FP,16419,Page 1 of 2 Compliments of: Bob Bartelme Backw...,"Based on the provided passages, here is how yo...",truncated,,
4,FP,7231,"\n{'name': 'Eastside Deli', 'address': '10068 ...",Eastside Deli is a local business located at 1...,7:00 AM to 8:00 PM on weekends,,


In [ ]:
error_df.to_csv(
    "lettucedetect_error_analysis_100.csv",
    index=False
)

print("Saved!")

Saved!


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
error_df.to_csv(
    "/content/drive/MyDrive/lettucedetect_error_analysis_100.csv",
    index=False
)

print("Saved to Google Drive!")

Saved to Google Drive!


In [ ]:
import os

path = "/content/drive/MyDrive/lettucedetect_error_analysis_100.csv"

print("File exists:", os.path.exists(path))

File exists: True


In [ ]:
!pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd

file_path = "/content/drive/MyDrive/categorized_errors.csv"

df = pd.read_csv(file_path)

print("CSV loaded successfully!")
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

df.head()

CSV loaded successfully!
Rows: 100
Columns: ['error_type', 'example_id', 'context', 'answer', 'problem_span', 'ragtruth_type', 'reason', 'category']


,error_type,example_id,context,answer,problem_span,ragtruth_type,reason,category
0,FP,13532,One difference between the FEHA (the Fair Empl...,"Based on the given passages, we can see that t...",twenty or more employees,NaN,NaN,Wrong occurrence / partial match
1,FP,6277,"\n{'name': 'Mala Town', 'address': '6555 Parda...",Mala Town is a local Chinese restaurant locate...,to,NaN,NaN,Boundary/tokenization artifact
2,FP,1112,An American citizen was wounded by gunfire Thu...,An American woman working at a medical school ...,The attack occurred despite a recent surge in...,NaN,NaN,Unsupported/fabricated (not annotated by gold)
3,FP,16419,Page 1 of 2 Compliments of: Bob Bartelme Backw...,"Based on the provided passages, here is how yo...",truncated,NaN,NaN,False alarm on supported content
4,FP,7231,"\n{'name': 'Eastside Deli', 'address': '10068 ...",Eastside Deli is a local business located at 1...,7:00 AM to 8:00 PM on weekends,NaN,NaN,Numbers/values mismatch


In [ ]:
for i in range(len(all_predictions)):
    answer = dataset["test"][i]["output"]
    for pred in all_predictions[i]:
        if pred["end"] > len(answer):
            print(f"Example {i} (id={dataset['test'][i]['id']}): "
                  f"pred end={pred['end']} exceeds len(answer)={len(answer)}")
            print(f"  pred text: {pred['text']!r}")
            print(f"  answer[pred['start']:pred['end']]: "
                  f"{answer[pred['start']:pred['end']]!r}")
            print()